# Part 1 — Document Extraction & Prompt Engineering Demo

This notebook walks through every phase of the Part 1 pipeline interactively:

1. PDF inspection — what pdfplumber actually produces
2. Table structure verification
3. Naive vs improved prompt comparison
4. Full pipeline run with audit evidence

> **Requires** `.env` with `GEMINI_API_KEY` set.

In [ ]:
import sys, os
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

PDF_PATH = "../data/fy2024_analysis_of_revenue_and_expenditure.pdf"
print("Environment loaded.  Model:", os.environ.get("GEMINI_MODEL", "gemini-2.5-flash"))

---
## 1. PDF inspection

In [ ]:
from src.part1.pdf_parser import get_page_text, get_pages_text, extract_table_page8, extract_table_page20

page5 = get_page_text(PDF_PATH, 5)
page6 = get_page_text(PDF_PATH, 6)

print("=== PAGE 5 (first 800 chars) ===")
print(page5[:800])

In [ ]:
print("=== PAGE 6 (first 600 chars) ===")
print(page6[:600])

---
## 2. Table inspection — pages 8 and 20

In [ ]:
table8 = extract_table_page8(PDF_PATH)

print("Table 8 title  :", table8.title)
print("Table 8 headers:", table8.headers)
print()
print("=== LLM-ready serialisation ===")
print(table8.to_llm_text())

In [ ]:
table20 = extract_table_page20(PDF_PATH)

print("Table 20 title  :", table20.title)
print("Table 20 headers:", table20.headers)
print()
print("=== LLM-ready serialisation ===")
print(table20.to_llm_text())

---
## 3. Naive vs improved prompt comparison

Both prompts are run against the same page-5 text.  The improved prompt returns a
structured `CorporateTaxEvidence` object with verbatim evidence that can be
deterministically verified against the source.  The naive prompt returns unstructured
prose that cannot be validated programmatically.

In [ ]:
import json
from src.part1.extractor import build_llm, demonstrate_naive_vs_improved

llm = build_llm()
comparison = demonstrate_naive_vs_improved(page5, llm)

for key in ("naive", "improved"):
    entry = comparison[key]
    print(f"\n{'='*60}")
    print(f"  {key.upper()} PROMPT")
    print(f"{'='*60}")
    print(f"  Prompt        : {entry['prompt']}")
    print(f"  Response type : {entry['response_type']}")
    print(f"  Structured    : {entry['structured']}")
    print(f"  Verified      : {entry['evidence_verified']}")
    print(f"  Notes         : {entry['notes']}")
    print(f"  Response      :")
    resp = entry['response']
    if isinstance(resp, dict):
        print(json.dumps(resp, indent=4))
    else:
        print(str(resp)[:600])

In [ ]:
print("\n=== COMPARISON DIMENSIONS ===")
for dim, desc in comparison["comparison"].items():
    print(f"\n  {dim}:\n    {desc}")

---
## 4. Full pipeline run

In [ ]:
from src.part1.extractor import run_extraction

result, evidence = run_extraction(PDF_PATH, llm)

print("\n=== PART 1 RESULT ===")
print(json.dumps(result.model_dump(), indent=2))

In [ ]:
print("\n=== EVIDENCE AUDIT TRAIL ===")
for ev in evidence:
    d = ev.model_dump()
    print(f"\nField            : {d['field_name']}")
    print(f"  Source page    : {d['source_page']}")
    print(f"  Source evidence: {d['source_evidence'][:120]}")
    print(f"  Raw value      : {d['raw_value']}")
    print(f"  Source unit    : {d['source_unit']}")
    print(f"  Normalized     : {d['normalized_value']}")
    print(f"  Validated      : {d['validation_passed']}")
    if d['validation_note']:
        print(f"  Note           : {d['validation_note']}")

In [ ]:
# Save outputs
import pathlib

out = pathlib.Path("../outputs")
out.mkdir(exist_ok=True)

(out / "part1_result.json").write_text(
    json.dumps(result.model_dump(), indent=2, ensure_ascii=False), encoding="utf-8"
)
(out / "part1_evidence.json").write_text(
    json.dumps([ev.model_dump() for ev in evidence], indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Outputs saved to ../outputs/")